# 02 — Swap the black box

**Companion to Lesson 4.**

In notebook 01 you measured a lexical (BM25) retriever on a
BEIR dataset. Now we'll keep everything else fixed — same
dataset, same queries, same metrics — and swap out the
retrieval strategy. This is the experimental discipline that
lets you say something meaningful when comparing systems:
change one variable, hold everything else constant.

## The three strategies

| name | how it works | strength |
|---|---|---|
| **lexical (BM25)** | matches the *words* in the query against the *words* in each doc | exact terms, names, IDs |
| **vector** | embeds the query and each doc; ranks by cosine similarity | semantic match, paraphrase |
| **hybrid** | runs both and fuses the rankings (weighted RRF) | the best of both |

Lexical is what powered search before embeddings; vector is the
modern semantic approach (today's reference point is **Voyage's
`voyage-context-3`**, which embeds each chunk *in the context
of its parent document* — see notebook 00 for the ingest
pipeline). Hybrid is the production default at most companies
because the two signals are complementary.

In [ ]:
import os, sys
# Notebooks live in agent-notebooks/. Add the repo root to the path so
# we can import the lab's library modules.
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

In [ ]:
# Match this to whatever you ingested.
DATASET   = "scifact"
N_QUERIES = 30
TOP_K     = 10

import pymongo
import voyageai
from lib import (
    MONGODB_BASE_URL, MONGODB_URI, DB_NAME, VOYAGE_API_KEY,
    collection_name, embed_queries, load_beir_dataset,
)
from retrieve import text_only, vector_only, hybrid
from lib_metrics import compute_query_metrics, aggregate_metrics, format_summary

client     = pymongo.MongoClient(MONGODB_URI)
coll       = client[DB_NAME][collection_name(DATASET)]
voyage     = voyageai.Client(api_key=VOYAGE_API_KEY, base_url=MONGODB_BASE_URL)
corpus, queries, qrels, info = load_beir_dataset(DATASET)

query_ids = list(queries.keys())[:N_QUERIES]
q_texts   = [queries[qid] for qid in query_ids]
q_vecs    = embed_queries(voyage, q_texts)
print(f"Embedded {len(q_vecs)} queries with voyage-3-large "
      f"({len(q_vecs[0])} dims).")

## Run all three strategies

We'll evaluate each strategy on the same set of queries and
collect aggregate metrics in a single table.

In [ ]:
STRATEGIES = {
    "lexical (BM25)" : lambda q_vec, q_text: text_only(coll, q_text, top_k=TOP_K),
    "vector"         : lambda q_vec, q_text: vector_only(coll, q_vec, top_k=TOP_K),
    "hybrid α=0.8"   : lambda q_vec, q_text: hybrid(coll, q_vec, q_text, top_k=TOP_K, alpha=0.8),
}

rows = {}
for name, run in STRATEGIES.items():
    per_query = []
    for qid, q_vec, q_text in zip(query_ids, q_vecs, q_texts):
        ranked = run(q_vec, q_text)
        ranked_ids = [r['doc_id'] for r in ranked]
        per_query.append(compute_query_metrics(ranked_ids, qrels.get(qid, {})))
    rows[name] = aggregate_metrics(per_query)
    print(f"{name:<20}  {format_summary(rows[name])}")

## Compare side-by-side

Same data in tabular form so the deltas are visible at a glance.

In [ ]:
import pandas as pd
df = pd.DataFrame(rows).T[["P@5", "R@5", "NDCG@5", "NDCG@10", "MRR", "MAP"]]
df.round(3)

## Hybrid weight `α` — controlling the blend

The `hybrid(..., alpha=0.8)` call passes a weight that controls
how much the **vector** ranking matters relative to the
**lexical** one:

- `alpha = 1.0` — vector only (equivalent to `vector_only`).
- `alpha = 0.0` — text only (equivalent to `text_only`).
- `alpha = 0.5` — naïve uniform fusion. *Often the wrong
  default* — it drags vector quality down when vector is the
  stronger signal.
- `alpha = 0.8` — vector-favored. A reasonable starting point
  on most BEIR datasets where embeddings dominate.

Let's sweep `α` to see how aggregate NDCG@10 moves.

In [ ]:
ALPHAS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
sweep_ndcg = []
for a in ALPHAS:
    per_query = []
    for qid, q_vec, q_text in zip(query_ids, q_vecs, q_texts):
        ranked = hybrid(coll, q_vec, q_text, top_k=TOP_K, alpha=a)
        ranked_ids = [r['doc_id'] for r in ranked]
        per_query.append(compute_query_metrics(ranked_ids, qrels.get(qid, {})))
    agg = aggregate_metrics(per_query)
    sweep_ndcg.append(agg["NDCG@10"])
    print(f"  α={a:.1f}  NDCG@10={agg['NDCG@10']:.3f}  MAP={agg['MAP']:.3f}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ALPHAS, sweep_ndcg, marker="o")
ax.set_xlabel("hybrid α  (0 = lexical, 1 = vector)")
ax.set_ylabel("NDCG@10")
ax.set_title(f"{DATASET} — NDCG@10 vs hybrid alpha ({N_QUERIES} queries)")
ax.grid(alpha=0.3)
plt.show()

## Reading the comparison

Three things to look for:

1. **Does vector beat lexical on NDCG@10?** On most modern
   BEIR datasets, yes — embeddings handle paraphrase and
   topical overlap that BM25 misses. But it's not universal:
   datasets dominated by exact strings (codes, names, IDs)
   still favour lexical.

2. **Does hybrid beat both?** Often yes, but only if `α` is
   weighted toward the stronger of the two single-mode
   strategies. The uniform `α=0.5` is the classic "hybrid
   hurts" trap — it drags vector quality down when vector is
   already the better signal.

3. **Which metric improves most?** If `vector` and `hybrid`
   both lift Recall@10 a lot more than NDCG@10, it means
   they're *finding* more relevant docs but not necessarily
   putting them at the top. That's the signal that a
   **reranker** would help (see `phase4/` for a cross-encoder
   rerank stage).

## What this is NOT telling you

BEIR's queries are **homogeneous within a dataset** — all
scientific claims, or all health questions. A single
well-chosen retrieval setup wins almost every query in such
a benchmark. Real production traffic looks nothing like that
— you'll get a mix of exact-match lookups, single-word topic
queries, multi-hop compound questions, and chatty natural
language all from the same users. Per-query routing /
rewriting / reranking comes into its own there. See
`phase4/query_classifier.py` for an example router.

## Next

Open **`03_curate_eval_set.ipynb`** — BEIR is great for
benchmarks but rarely matches your real users' queries. To
know which strategy works in *your* context, you need *your
own* eval data.